<a href="https://colab.research.google.com/github/sanikadoes/fedex-mumbai-lastmile/blob/main/fedex_mumbai_lastmile.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [28]:
#import the required packages

import osmnx as ox
import networkx as nx
import pickle

# download road data from OpenStreetMap for Andheri, Bandra, Colaba
# these areas are selected due to their high network density

zones = {
    'Andheri': 'Andheri, Mumbai',
    'Bandra':  'Bandra, Mumbai',
    'Colaba':   'Colaba, Mumbai'
}

networks = {}

for zone_name, zone_query in zones.items():
    print(f"Downloading {zone_name}...")
    try:
        #geocode the place to get coordinates
        point = ox.geocode(zone_query)
        #use graph_from_point with a 1 km radius
        G = ox.graph_from_point(point, dist=1000, network_type='drive')
        networks[zone_name] = G
        print(f"{zone_name}: {len(G.nodes)} intersections, {len(G.edges)} road segments")
    except Exception as e:
        print(f"Failed to download {zone_name}: {e}")
        continue #continue to the next zone if one fails

#save the file locally because downloading takes time
#pickle is able to freeze the map data into a file
with open('mumbai_networks.pkl', 'wb') as f:
    pickle.dump(networks, f)

print("Networks saved.")

Andheri: 600 intersections, 1263 road segments
Bandra: 511 intersections, 1116 road segments
Colaba: 223 intersections, 458 road segments
Networks saved.


In [29]:
#now we generate the delivery stops
#they are not real - they are simulated
#for this project, we will simulate 15 delivery stops per location

import osmnx as ox
import pandas as pd
import numpy as np
import sqlite3
import pickle
import random
import os

#load the saved network
with open('mumbai_networks.pkl', 'rb') as f:
    networks = pickle.load(f)

random.seed(42)

all_stops = []

for zone_name, G in networks.items():

    #get all nodes in this zone
    nodes = list(G.nodes(data=True))

    #randomly sample 15 delivery stops per zone
    sampled = random.sample(nodes, 15)

    #save this into a data frame
    for i, (node_id, data) in enumerate(sampled):
        all_stops.append({
            'stop_id':       f"{zone_name[:3].upper()}-{i+1:03d}",
            'zone':           zone_name,
            'node_id':        node_id,
            'lat':            data['y'],
            'lon':            data['x'],
            'address':        f"Delivery Point {i+1}, {zone_name}, Mumbai",
            'package_count':  random.randint(1, 8),
            'time_window_start': random.choice([9, 10, 11, 12]),
            'time_window_end':   random.choice([14, 15, 16, 17, 18]),
            'priority':       random.choice(['Standard', 'Standard',
                                             'Standard', 'Express'])
        })

stops_df = pd.DataFrame(all_stops)

#save to SQLite
conn = sqlite3.connect('fedex_mumbai.db')
stops_df.to_sql('delivery_stops', conn, if_exists='replace', index=False)
conn.close()

#making sure that the directory exists
os.makedirs('data/clean', exist_ok=True)

#csv backup
stops_df.to_csv('data/clean/delivery_stops.csv', index=False)

#print the result
print(stops_df.groupby('zone')[['stop_id','package_count']].agg(
    stops=('stop_id','count'),
    total_packages=('package_count','sum')
))

         stops  total_packages
zone                          
Andheri     15              58
Bandra      15              65
Colaba      15              66


In [30]:
import osmnx as ox
import networkx as nx
import pandas as pd
import numpy as np
import sqlite3
import pickle

#load networks and stops
with open('mumbai_networks.pkl', 'rb') as f:
    networks = pickle.load(f)

conn = sqlite3.connect('fedex_mumbai.db')
stops_df = pd.read_sql('SELECT * FROM delivery_stops', conn)
conn.close()

In [31]:
#travelling salesman problem
#we use the greedy logic: nearest neighbour
#we start at a point, look at all unvisited stops
#we select the closest one
#repeat until all stops are visited

#shortest path algo
def get_route_distance(G, node_a, node_b):
    try:
        return nx.shortest_path_length(G, node_a, node_b, weight='length')
    except (nx.NetworkXNoPath, nx.NodeNotFound):
        return np.inf

#building the distance matrix
def build_distance_matrix(G, nodes):
    n = len(nodes)
    matrix = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            if i != j:
                matrix[i][j] = get_route_distance(G, nodes[i], nodes[j])
    return matrix

#nearest neighbour route
def nearest_neighbour_route(distance_matrix, starts=None):
    """
    Try multiple starting points, return the shortest route found.
    Defaults to trying all stops as starting point if n <= 15.
    """
    n = len(distance_matrix)
    if starts is None:
        starts = range(n)  #try every stop as starting point

    best_route    = None
    best_distance = np.inf

    for start in starts:
        unvisited = [i for i in range(n) if i != start]
        route = [start]

        while unvisited:
            current   = route[-1]
            reachable = [x for x in unvisited
                         if not np.isinf(distance_matrix[current][x])]

            if not reachable:
                route.extend(unvisited)
                break

            nearest = min(reachable,
                          key=lambda x: distance_matrix[current][x])
            route.append(nearest)
            unvisited.remove(nearest)

        distance = calculate_total_distance(distance_matrix, route)

        if distance < best_distance:
            best_distance = distance
            best_route    = route

    return best_route

def calculate_total_distance(distance_matrix, route):
    total = 0
    for i in range(len(route) - 1):
        d = distance_matrix[route[i]][route[i+1]]
        if not np.isinf(d):
            total += d
    return total

#confirm that nodes actually exist in each graph

print("NODES")
for zone_name, G in networks.items():
    zone_stops = stops_df[stops_df['zone'] == zone_name].copy()
    #re-snap fresh against this zone's graph
    snapped_nodes = [
        ox.nearest_nodes(G, row['lon'], row['lat'])
        for _, row in zone_stops.iterrows()
    ]
    missing = [n for n in snapped_nodes if n not in G.nodes]
    print(f"{zone_name}: {len(snapped_nodes)} stops | "
          f"{len(missing)} nodes missing from graph")
print("--------------------\n")


#routing loop

results = []

for zone_name, G in networks.items():
    zone_stops = stops_df[stops_df['zone'] == zone_name].copy()

    #re-snap directly against THIS zone's graph
    nodes = [
        ox.nearest_nodes(G, row['lon'], row['lat'])
        for _, row in zone_stops.iterrows()
    ]

    print(f"processing {zone_name} ({len(nodes)} stops)...")

    #skip zone if any nodes are missing
    missing = [n for n in nodes if n not in G.nodes]
    if missing:
        print(f"skipped {len(missing)} nodes not found in graph.\n")
        continue

    #build distance matrix
    dist_matrix = build_distance_matrix(G, nodes)

    #report unreachable pairs
    inf_count = int(np.sum(np.isinf(dist_matrix)))
    print(f"unreachable pairs in matrix: {inf_count}")

    #naive route: visit stops in original order
    naive_route     = list(range(len(nodes)))
    naive_distance  = calculate_total_distance(dist_matrix, naive_route)

    #optimized route: nearest neighbour
    optimized_route    = nearest_neighbour_route(dist_matrix)
    optimized_distance = calculate_total_distance(dist_matrix, optimized_route)

    #guard against division by zero - thanks Codex
    if naive_distance == 0:
        print(f"WARNING!! naive distance is 0, skip\n")
        continue

    improvement_pct = ((naive_distance - optimized_distance) /
                        naive_distance * 100)

    results.append({
        'zone':                  zone_name,
        'stops':                 len(nodes),
        'naive_distance_km':     round(naive_distance / 1000, 2),
        'optimized_distance_km': round(optimized_distance / 1000, 2),
        'distance_saved_km':     round((naive_distance - optimized_distance) / 1000, 2),
        'improvement_pct':       round(improvement_pct, 1),
        'naive_route':           naive_route,
        'optimized_route':       optimized_route,
        'nodes':                 nodes,
        'zone_name':             zone_name
    })

    print(f"  naive:     {naive_distance/1000:.2f} km")
    print(f"  optimized: {optimized_distance/1000:.2f} km")
    print(f"  saved:     {improvement_pct:.1f}%\n")


#save all the results

if results:
    results_df = pd.DataFrame([
        {k: v for k, v in r.items()
         if k not in ['naive_route', 'optimized_route', 'nodes', 'zone_name']}
        for r in results
    ])

    conn = sqlite3.connect('fedex_mumbai.db')
    results_df.to_sql('route_results', conn, if_exists='replace', index=False)
    conn.close()

    print("\nresults")
    print(results_df.to_string(index=False))
else:
    print("no results!! all zones were skipped.")

NODES
Andheri: 15 stops | 0 nodes missing from graph
Bandra: 15 stops | 0 nodes missing from graph
Colaba: 15 stops | 0 nodes missing from graph
--------------------

processing Andheri (15 stops)...
unreachable pairs in matrix: 28
  naive:     17.25 km
  optimized: 7.69 km
  saved:     55.4%

processing Bandra (15 stops)...
unreachable pairs in matrix: 28
  naive:     25.76 km
  optimized: 8.15 km
  saved:     68.3%

processing Colaba (15 stops)...
unreachable pairs in matrix: 14
  naive:     15.96 km
  optimized: 5.50 km
  saved:     65.5%


results
   zone  stops  naive_distance_km  optimized_distance_km  distance_saved_km  improvement_pct
Andheri     15              17.25                   7.69               9.56             55.4
 Bandra     15              25.76                   8.15              17.61             68.3
 Colaba     15              15.96                   5.50              10.45             65.5


In [32]:
import folium
import networkx as nx
import osmnx as ox

#colours for the interactive map

zone_colours = {
    'Andheri': '#E63946',
    'Bandra':  '#2196F3',
    'Colaba':  '#4CAF50'
}


#get the actual road co0ordinates

def get_actual_route_coords(G, node_a, node_b):
    try:
        path_nodes = nx.shortest_path(G, node_a, node_b, weight='length')
        return [(G.nodes[n]['y'], G.nodes[n]['x']) for n in path_nodes]
    except (nx.NetworkXNoPath, nx.NodeNotFound):
        return []


#map 1: all delivery stops across all zones

mumbai_map = folium.Map(
    location=[19.0760, 72.8777],
    zoom_start=12,
    tiles='CartoDB positron'
)

for _, stop in stops_df.iterrows():
    folium.CircleMarker(
        location=[stop['lat'], stop['lon']],
        radius=9 if stop['priority'] == 'Express' else 6,
        color=zone_colours[stop['zone']],
        fill=True,
        fill_color=zone_colours[stop['zone']],
        fill_opacity=0.85,
        popup=folium.Popup(
            f"<b>{stop['stop_id']}</b><br>"
            f"Zone: {stop['zone']}<br>"
            f"Priority: {stop['priority']}<br>"
            f"Packages: {stop['package_count']}<br>"
            f"Window: {stop['time_window_start']}:00–{stop['time_window_end']}:00",
            max_width=200
        )
    ).add_to(mumbai_map)

legend_html = """
<div style="position:fixed;bottom:30px;left:30px;z-index:1000;
     background:white;padding:15px;border-radius:8px;
     border:2px solid #ccc;font-family:Arial;font-size:13px;
     box-shadow:2px 2px 6px rgba(0,0,0,0.2);">
<b style="font-size:14px;">FedEx Mumbai — Delivery Zones</b><br><br>
<span style="color:#E63946;font-size:18px;">●</span> Andheri<br>
<span style="color:#2196F3;font-size:18px;">●</span> Bandra<br>
<span style="color:#4CAF50;font-size:18px;">●</span> Colaba<br><br>
<span style="font-size:11px;color:#666;">
  Large dot = Express | Small dot = Standard
</span>
</div>
"""
mumbai_map.get_root().html.add_child(folium.Element(legend_html))
mumbai_map.save('01_all_delivery_stops.html')
print("✓ Map 1 saved: 01_all_delivery_stops.html")


#map 2: optimized routes per location

for result in results:
    zone_name  = result['zone_name']
    G          = networks[zone_name]
    zone_stops = stops_df[stops_df['zone'] == zone_name].copy().reset_index(drop=True)
    nodes      = result['nodes']
    opt_route  = result['optimized_route']
    naive_route = result['naive_route']

    centre_lat = zone_stops['lat'].mean()
    centre_lon = zone_stops['lon'].mean()
    colour     = zone_colours[zone_name]

    zone_map = folium.Map(
        location=[centre_lat, centre_lon],
        zoom_start=14,
        tiles='CartoDB positron'
    )

    #naive route
    for i in range(len(naive_route) - 1):
        coords = get_actual_route_coords(
            G, nodes[naive_route[i]], nodes[naive_route[i+1]]
        )
        if coords:
            folium.PolyLine(
                locations=coords,
                color='#999999',
                weight=3,
                opacity=0.5,
                tooltip=f"Naive segment {i+1}→{i+2}"
            ).add_to(zone_map)

    #optimized route by zone colour
    for i in range(len(opt_route) - 1):
        coords = get_actual_route_coords(
            G, nodes[opt_route[i]], nodes[opt_route[i+1]]
        )
        if coords:
            folium.PolyLine(
                locations=coords,
                color=colour,
                weight=5,
                opacity=0.85,
                tooltip=f"Optimized segment {i+1}→{i+2}"
            ).add_to(zone_map)

    #optimized route: numbered stop markers
    for seq, stop_idx in enumerate(opt_route):
        if stop_idx >= len(zone_stops):
            continue
        stop_row = zone_stops.iloc[stop_idx]
        folium.Marker(
            location=[stop_row['lat'], stop_row['lon']],
            popup=folium.Popup(
                f"<b>Stop {seq+1}: {stop_row['stop_id']}</b><br>"
                f"Priority: {stop_row['priority']}<br>"
                f"Packages: {stop_row['package_count']}<br>"
                f"Window: {stop_row['time_window_start']}:00–"
                f"{stop_row['time_window_end']}:00",
                max_width=200
            ),
            icon=folium.DivIcon(
                html=(
                    f'<div style="background:{colour};color:white;'
                    f'border-radius:50%;width:26px;height:26px;'
                    f'text-align:center;line-height:26px;'
                    f'font-weight:bold;font-size:12px;'
                    f'border:2px solid white;'
                    f'box-shadow:1px 1px 3px rgba(0,0,0,0.3);">'
                    f'{seq+1}</div>'
                ),
                icon_size=(26, 26),
                icon_anchor=(13, 13)
            )
        ).add_to(zone_map)

    #legend
    summary_html = f"""
    <div style="position:fixed;top:15px;right:15px;z-index:1000;
         background:white;padding:14px 18px;border-radius:8px;
         border:2px solid #ddd;font-family:Arial;font-size:13px;
         box-shadow:2px 2px 6px rgba(0,0,0,0.15);">
    <b style="font-size:14px;">FedEx {zone_name}</b><br>
    <span style="color:#666;">Last-Mile Route Optimization</span><br><br>
    Stops: <b>{result['stops']}</b><br>
    Naive distance: <b>{result['naive_distance_km']} km</b><br>
    Optimized distance: <b>{result['optimized_distance_km']} km</b><br>
    <span style="color:green;font-weight:bold;font-size:15px;">
      ↓ {result['improvement_pct']}% saved
    </span><br><br>
    <span style="font-size:11px;color:#999;">
      ── Naive route &nbsp;&nbsp;
      <span style="color:{colour};">── Optimized route</span>
    </span>
    </div>
    """
    zone_map.get_root().html.add_child(folium.Element(summary_html))

    filename = f"02_route_{zone_name.lower()}.html"
    zone_map.save(filename)
    print(f"✓ Map 2 saved: {filename}")

print("\n all maps generated as HTML.")

✓ Map 1 saved: 01_all_delivery_stops.html
✓ Map 2 saved: 02_route_andheri.html
✓ Map 2 saved: 02_route_bandra.html
✓ Map 2 saved: 02_route_colaba.html

 all maps generated as HTML.
